[![Colabで開く](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/schwalbe1996/ds_media_intro/blob/main/chap15_ex.ipynb)

# 15章 演習問題（宿題）：MNIST用のCNNでFashion-MNIST（衣類画像）を認識しよう

このノートブックは 15 章「画像データの機械学習」の宿題です。今回は学習に時間がかかるので講義時間中には終わらないかもしれません。

このノートブックには **コードの一部しか書かれていません**。`chap15.ipynb` を開いて見ながら、`# TODO` の部分を**自分で埋めて**完成させましょう。「どのセルのコードが使えそうか」を探すことも勉強のうちです。

> **⚠️ 実行前に必ず GPU を有効化してください**：Colab メニューの「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータを **GPU** にする。CPU のままだと学習にとても時間がかかります。

## 課題：数字認識のCNNは、そのまま衣類画像にも使えるか？
`chap15.ipynb` では **手書き数字（MNIST）** を畳み込みニューラルネットワーク（CNN）で認識し、テスト精度は約 **98.3%** でした。
同じCNNの構造のまま、入力データだけを **Fashion-MNIST（10種類の衣類のモノクロ画像）** に差し替えて学習し、精度がどうなるかを調べます。

- Fashion-MNIST も「28×28 のモノクロ画像・10クラス」なので、**CNNの構造は一切変えずに**データだけ差し替えれば動きます。
- やることは2か所の `# TODO` を埋めるだけです。
  1. データセットを Fashion-MNIST に差し替える（`chap15.ipynb` の最後のセルが参考になります）
  2. 学習したモデルで予測クラスを求める（`chap15.ipynb` の「予測」のセルが参考になります）

**観察と考察:**
- Fashion-MNIST のテスト精度は、MNIST の 98.3% と比べて高い？低い？
- クラス別の正解率を見て、**特に間違えやすいクラス**はどれ？（似た形の衣類どうしに注目）
- なぜ数字より難しい（または易しい）のか、提出セルの `answer_A` に1〜3行で説明してください。

## 提出方法（重要）
最後の **「提出ファイルの作成」セル** を実行すると、結果（予測例・精度の推移・クラス別正解率）と考察を1枚にまとめた PDF が自動で作られ、ダウンロードされます。
- ファイル名は **`学籍番号_フルネーム_15.pdf`** になります。氏名は**ローマ字・名+姓・スペースなし**で書きます（例: 彦根太郎 → `TaroHikone`、ファイル名 → `5025123_TaroHikone_15.pdf`）。提出セルの `student_id` と `student_name` を**自分のものに書き換えてから**実行してください。
- 提出するのはこの **PDF 1ファイルだけ** です。

In [ ]:
# ライブラリの読み込み（このセルはそのまま実行）
!pip install -q japanize-matplotlib setuptools  # setuptools は japanize の依存(distutils)対策

import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import japanize_matplotlib  # PDF内の日本語を表示するため

## ステップ1：データセットを Fashion-MNIST にする

`chap15.ipynb` の **最後のセル**（`datasets.FashionMNIST(...)` が出てくるセル）を参考に、`# TODO` を埋めよう。

In [ ]:
batch_size = 32

# TODO: Fashion-MNIST のデータセットを読み込む（chap15.ipynb の最後のセルを参考に）
#       training_data（train=True）と test_data（train=False）の2つを作る。
#       どちらも download=True, transform=ToTensor() を指定すること。
# training_data = ...
# test_data = ...


# === ここから下はそのまま実行 ===
train_dataloader = DataLoader(training_data, batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size)

# Fashion-MNIST のクラス名（0〜9 が衣類の種類に対応する）
class_names = ['Tシャツ/トップス', 'ズボン', 'プルオーバー', 'ワンピース', 'コート',
               'サンダル', 'シャツ', 'スニーカー', 'バッグ', 'アンクルブーツ']
print('学習データ数:', len(training_data), ' テストデータ数:', len(test_data))

## ステップ2：CNN を定義する（このセルはそのまま実行）

`chap15.ipynb` の畳み込みニューラルネットワークと**まったく同じ構造**です。中身を変える必要はありません。
入力が「28×28・1チャンネル・10クラス」なので、数字でも衣類でもそのまま使えます。

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 6, 5),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(6, 16, 5),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc1 = nn.Sequential(nn.Linear(16*4*4, 120), nn.ReLU())
        self.fc2 = nn.Sequential(nn.Linear(120, 84), nn.ReLU())
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)
        return out

## ステップ3：学習する（このセルはそのまま実行）

`chap15.ipynb` の学習のセルとほぼ同じです。各エポックのテスト精度を `acc_history` に記録しています。
（GPU なら数分で終わります。CPU だと時間がかかるので、上の注意のとおり GPU を有効にしてください）

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用デバイス:", device)

model = MyModel().to(device)
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1.0e-4)

epochs = 5  # 余裕があれば増やしてみてもよい
acc_history = []
for epoch in range(epochs):
    train_loss = 0
    model.train()
    for X, y in train_dataloader:
        X = X.to(device); y = y.to(device)
        model.zero_grad()
        y_pred = model(X)
        loss = loss_func(y_pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_dataloader)

    # テスト精度を計算
    model.eval()
    correct = 0
    for X, y in test_dataloader:
        X = X.to(device); y = y.to(device)
        with torch.no_grad():
            y_pred = model(X)
        correct += (y_pred.argmax(dim=1) == y).sum().item()
    acc = correct / len(test_dataloader.dataset)
    acc_history.append(acc)
    print(f'Epoch {epoch}: Train Loss={train_loss:.4f}  Test Accuracy={acc:.4f}')

## ステップ4：予測を集めてクラス別の正解率を調べる

`chap15.ipynb` の **「予測」のセル**（`model(x).argmax(dim=1)` が出てくるセル）を参考に、`# TODO` を埋めよう。

In [ ]:
model.eval()
all_pred = []
all_true = []
for X, y in test_dataloader:
    X = X.to(device)
    with torch.no_grad():
        # TODO: モデルの出力から予測クラス pred を求める（chap15.ipynb の「予測」セルを参考に）
        #       ヒント: model(X) の出力に .argmax(dim=1) を使い、最後に .cpu() を付ける
        # pred = ...
        all_pred.append(pred)
    all_true.append(y)

# === ここから下はそのまま実行 ===
all_pred = torch.cat(all_pred).numpy()
all_true = torch.cat(all_true).numpy()

overall_acc = (all_pred == all_true).mean()
print(f'テスト全体の正解率: {overall_acc:.4f}')

class_acc = []
for c in range(10):
    mask = all_true == c
    class_acc.append((all_pred[mask] == c).mean())
    print(f'  {class_names[c]}: {class_acc[c]:.3f}')

## 提出ファイルの作成

下の `student_id`・`student_name`・`answer_A` を書き換えてから実行すると、提出用 PDF が作られてダウンロードされます。**この PDF 1ファイルだけ** を提出してください。

In [ ]:
# ★ 自分の情報に書き換える ★
student_id   = "5025000"             # 学籍番号（7桁）
student_name = "TaroHikone"          # フルネーム（ローマ字・名+姓・スペースなし。例: 彦根太郎→TaroHikone）

# 考察は下の """ と """ の間に書く（自動で折り返されます）
answer_A = """ここに考察を書く。
例）Fashion-MNIST の正解率は MNIST(98.3%) と比べて高い/低い？
どのクラスが間違えやすかった？ なぜ数字より難しい/易しいと考えられる？"""

# === ここから下はそのまま実行 ===
def wrap_jp(text, width=46):
    out = []
    for para in text.split("\n"):
        if para == "":
            out.append("")
        else:
            out += [para[i:i+width] for i in range(0, len(para), width)]
    return "\n".join(out)

# 表示用にテスト画像を1バッチ取得して予測
x_show, y_show = next(iter(test_dataloader))
model.eval()
with torch.no_grad():
    pred_show = model(x_show.to(device)).argmax(dim=1).cpu()

fig = plt.figure(figsize=(8.27, 11.69))  # A4縦
gs = gridspec.GridSpec(3, 1, height_ratios=[1.3, 1.0, 0.8])
fig.suptitle(f"{student_id} {student_name}　マルチメディア処理入門 宿題15", fontsize=14)

# 上段：予測結果の例（10枚）。正解は黒、誤りは赤
gs_top = gs[0].subgridspec(2, 5, hspace=0.6, wspace=0.1)
for i in range(10):
    ax = fig.add_subplot(gs_top[i // 5, i % 5])
    ax.imshow(x_show[i, 0], cmap='gray')
    ok = (pred_show[i].item() == y_show[i].item())
    ax.set_title(f"予:{class_names[pred_show[i]]}\n正:{class_names[y_show[i]]}",
                 fontsize=6, color=('black' if ok else 'red'))
    ax.axis('off')

# 中段左：テスト精度の推移
gs_mid = gs[1].subgridspec(1, 2, wspace=0.5)
ax_curve = fig.add_subplot(gs_mid[0])
ax_curve.plot(range(len(acc_history)), acc_history, marker='o')
ax_curve.set_title('テスト精度の推移'); ax_curve.set_xlabel('epoch'); ax_curve.set_ylabel('正解率')
ax_curve.set_ylim(0, 1); ax_curve.grid(True, alpha=0.3)

# 中段右：クラス別の正解率
ax_bar = fig.add_subplot(gs_mid[1])
ax_bar.barh(range(10), class_acc)
ax_bar.set_yticks(range(10)); ax_bar.set_yticklabels(class_names, fontsize=7)
ax_bar.set_xlim(0, 1); ax_bar.invert_yaxis(); ax_bar.set_title('クラス別の正解率')

# 下段：考察
ax_txt = fig.add_subplot(gs[2]); ax_txt.axis('off')
ax_txt.text(0.01, 0.99, f"テスト全体の正解率: {overall_acc:.3f}\n\n【考察】\n" + wrap_jp(answer_A),
            va='top', ha='left', fontsize=10)

filename = f"{student_id}_{student_name}_15.pdf"
fig.savefig(filename, bbox_inches='tight')
plt.show()
print("保存しました:", filename)

try:
    from google.colab import files
    files.download(filename)
except Exception:
    print("（Colab 以外では自動ダウンロードされません。左のファイル一覧から取得してください）")